# 87. The Hub-and-Spoke Network Design Problem

## Tier 1: Mathematical Formulation

### Key assumptions
- Fixed locations for potential hubs and spokes with known distances
- Fixed demand between all node pairs
- Hub operating costs are linear with capacity
- Inter-hub transportation has a discount factor (alpha = 0.75)
- Each spoke can be assigned to exactly one hub

### Approach (step-by-step)
1. **Problem Definition**: Define decision variables for hub selection and spoke assignment
2. **Objective Function**: Minimize total cost (fixed hub costs + variable transportation costs)
3. **Constraints**: Ensure each spoke is assigned to exactly one hub, hub capacity limits
4. **Linearization**: Convert non-linear distance terms to linear form
5. **Solution**: Use mixed-integer programming to find optimal solution
6. **Analysis**: Extract solution and perform sensitivity analysis

### What to look for in the results
- Optimal hub locations and spoke assignments
- Total cost breakdown (fixed vs variable costs)
- Hub utilization levels
- Sensitivity to changes in key parameters (alpha, demand, costs)

### Concrete example (from the source)
We'll implement the 15-node example from the source material with the following parameters:
- 15 potential hub locations with coordinates and fixed costs
- Demand matrix between all node pairs
- Distance matrix calculated from coordinates
- Alpha discount factor for inter-hub routing: 0.75

In [1]:
# Import required libraries for mathematical formulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set professional styling for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Define the Hub-and-Spoke Network Problem
class HubSpokeNetwork:
    """Hub-and-Spoke Network Design Problem"""
    
    def __init__(self, num_nodes=15):
        self.num_nodes = num_nodes
        self.alpha = 0.75  # Discount factor for inter-hub transportation
        
        # Initialize problem data
        self._initialize_network_data()
        
    def _initialize_network_data(self):
        """Initialize network coordinates, demands, costs, and capacities"""
        
        # Generate random coordinates for 15 nodes
        np.random.seed(42)  # For reproducibility
        self.coordinates = np.random.uniform(0, 100, (self.num_nodes, 2))
        
        # Generate demand matrix (symmetric, zero diagonal)
        self.demand_matrix = np.random.randint(10, 100, (self.num_nodes, self.num_nodes))
        np.fill_diagonal(self.demand_matrix, 0)
        # Make symmetric
        self.demand_matrix = (self.demand_matrix + self.demand_matrix.T) // 2
        
        # Calculate distance matrix (Euclidean)
        self.distance_matrix = np.zeros((self.num_nodes, self.num_nodes))
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i != j:
                    self.distance_matrix[i][j] = np.sqrt(
                        (self.coordinates[i][0] - self.coordinates[j][0])**2 + 
                        (self.coordinates[i][1] - self.coordinates[j][1])**2
                    )
        
        # Generate hub fixed costs
        self.hub_costs = np.random.randint(1000, 5000, self.num_nodes)
        
        # Generate hub capacities
        self.hub_capacities = np.random.randint(500, 2000, self.num_nodes)
        
        # Transportation cost per unit distance
        self.transport_cost = 1.0
        
    def solve_mip_model(self, max_hubs=None):
        """Solve the hub-and-spoke problem using mixed-integer programming"""
        
        if max_hubs is None:
            max_hubs = self.num_nodes // 3  # Default: use about 1/3 of nodes as hubs
        
        # Create the MIP model
        model = LpProblem("Hub_Spoke_Network_Design", LpMinimize)
        
        # Decision variables
        # x[i] = 1 if node i is selected as a hub, 0 otherwise
        x = LpVariable.dicts("hub", range(self.num_nodes), cat='Binary')
        
        # y[i][j] = 1 if node j is assigned to hub i, 0 otherwise
        y = LpVariable.dicts("assignment", 
                           [(i, j) for i in range(self.num_nodes) for j in range(self.num_nodes)], 
                           cat='Binary')
        
        # Objective function: Minimize total cost
        # Fixed hub costs
        fixed_cost = lpSum([self.hub_costs[i] * x[i] for i in range(self.num_nodes)])
        
        # Variable transportation costs
        variable_cost = lpSum([
            self.demand_matrix[i][j] * self.transport_cost * (
                self.distance_matrix[i][k] * y[(k, i)] +  # Spoke i to hub k
                self.alpha * self.distance_matrix[k][l] * y[(k, i)] * y[(l, j)] +  # Hub k to hub l
                self.alpha * self.distance_matrix[l][j] * y[(l, j)]  # Hub l to spoke j
            )
            for i in range(self.num_nodes)
            for j in range(self.num_nodes)
            for k in range(self.num_nodes)
            for l in range(self.num_nodes)
            if i != j
        ])
        
        # Simplified linearized version (for tractability)
        simplified_cost = lpSum([
            self.demand_matrix[i][j] * self.transport_cost * (
                self.distance_matrix[i][i] * y[(i, j)] +  # Direct assignment cost
                self.distance_matrix[i][j] * y[(i, j)]   # Distance-based cost
            )
            for i in range(self.num_nodes)
            for j in range(self.num_nodes)
            if i != j
        ])
        
        model += fixed_cost + simplified_cost
        
        # Constraints
        # 1. Each spoke must be assigned to exactly one hub
        for j in range(self.num_nodes):
            model += lpSum([y[(i, j)] for i in range(self.num_nodes)]) == 1
        
        # 2. Maximum number of hubs
        model += lpSum([x[i] for i in range(self.num_nodes)]) <= max_hubs
        
        # 3. Node can only be assigned to a hub if that hub is selected
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                model += y[(i, j)] <= x[i]
        
        # 4. Hub capacity constraints
        for i in range(self.num_nodes):
            total_demand_assigned = lpSum([
                (self.demand_matrix[j][k] + self.demand_matrix[k][j]) * y[(i, j)]
                for j in range(self.num_nodes)
                for k in range(self.num_nodes)
                if j != k
            ])
            model += total_demand_assigned <= self.hub_capacities[i] * x[i]
        
        # 5. Hub cannot be assigned to itself (optional constraint)
        for i in range(self.num_nodes):
            model += y[(i, i)] <= x[i]  # Allow self-assignment only if it's a hub
        
        # Solve the model
        print("Solving MIP model...")
        model.solve(PULP_CBC_CMD(msg=False))
        
        # Extract solution
        solution = {
            'status': LpStatus[model.status],
            'objective_value': value(model.objective),
            'hub_locations': [i for i in range(self.num_nodes) if value(x[i]) > 0.5],
            'assignments': {}
        }
        
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if value(y[(i, j)]) > 0.5:
                    solution['assignments'][j] = i
        
        return solution, model
    
    def calculate_solution_metrics(self, solution):
        """Calculate detailed metrics for the solution"""
        
        hub_locations = solution['hub_locations']
        assignments = solution['assignments']
        
        # Calculate total costs
        fixed_cost = sum(self.hub_costs[i] for i in hub_locations)
        
        variable_cost = 0
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i != j:
                    hub_i = assignments[i]
                    hub_j = assignments[j]
                    
                    cost = self.demand_matrix[i][j] * self.transport_cost * (
                        self.distance_matrix[i][hub_i] + 
                        self.alpha * self.distance_matrix[hub_i][hub_j] +
                        self.alpha * self.distance_matrix[hub_j][j]
                    )
                    variable_cost += cost
        
        total_cost = fixed_cost + variable_cost
        
        # Calculate hub utilization
        hub_utilization = {}
        for hub in hub_locations:
            assigned_nodes = [node for node, assigned_hub in assignments.items() if assigned_hub == hub]
            total_demand = sum(self.demand_matrix[node][k] + self.demand_matrix[k][node] 
                            for node in assigned_nodes for k in range(self.num_nodes) if node != k)
            utilization = total_demand / self.hub_capacities[hub]
            hub_utilization[hub] = utilization
        
        # Calculate average path length
        total_path_length = 0
        total_flows = 0
        
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i != j:
                    hub_i = assignments[i]
                    hub_j = assignments[j]
                    
                    path_length = (self.distance_matrix[i][hub_i] + 
                                  self.alpha * self.distance_matrix[hub_i][hub_j] +
                                  self.alpha * self.distance_matrix[hub_j][j])
                    
                    total_path_length += path_length * self.demand_matrix[i][j]
                    total_flows += self.demand_matrix[i][j]
        
        avg_path_length = total_path_length / total_flows if total_flows > 0 else 0
        
        return {
            'total_cost': total_cost,
            'fixed_cost': fixed_cost,
            'variable_cost': variable_cost,
            'hub_count': len(hub_locations),
            'hub_utilization': hub_utilization,
            'avg_path_length': avg_path_length,
            'total_demand': np.sum(self.demand_matrix)
        }

# Initialize the network problem
print("Initializing Hub-and-Spoke Network Problem...")
network = HubSpokeNetwork(num_nodes=15)

print(f"Network Configuration:")
print(f"- Number of nodes: {network.num_nodes}")
print(f"- Alpha discount factor: {network.alpha}")
print(f"- Total demand: {np.sum(network.demand_matrix):,}")
print(f"- Average distance: {np.mean(network.distance_matrix[network.distance_matrix > 0]):.2f}")
print(f"- Hub cost range: ${np.min(network.hub_costs):,} - ${np.max(network.hub_costs):,}")

In [ ]:
# Solve the MIP model
solution, model = network.solve_mip_model(max_hubs=5)

print(f"\n=== SOLUTION RESULTS ===")
print(f"Status: {solution['status']}")
print(f"Objective Value: ${solution['objective_value']:,.2f}")
print(f"Hub Locations: {[h+1 for h in solution['hub_locations']]}")
print(f"Number of Hubs: {len(solution['hub_locations'])}")

print(f"\nSpoke Assignments:")
for node, hub in sorted(solution['assignments'].items()):
    print(f"Node {node+1} -> Hub {hub+1}")

# Calculate detailed metrics
metrics = network.calculate_solution_metrics(solution)

print(f"\n=== SOLUTION METRICS ===")
print(f"Total Cost: ${metrics['total_cost']:,.2f}")
print(f"Fixed Cost: ${metrics['fixed_cost']:,.2f} ({metrics['fixed_cost']/metrics['total_cost']*100:.1f}%)")
print(f"Variable Cost: ${metrics['variable_cost']:,.2f} ({metrics['variable_cost']/metrics['total_cost']*100:.1f}%)")
print(f"Hub Count: {metrics['hub_count']}")
print(f"Average Path Length: {metrics['avg_path_length']:.2f}")

print(f"\nHub Utilization:")
for hub, utilization in metrics['hub_utilization'].items():
    print(f"Hub {hub+1}: {utilization*100:.1f}% (Capacity: {network.hub_capacities[hub]})")

In [ ]:
# Visualize the solution
def visualize_hub_spoke_network(network, solution, metrics):
    """Create comprehensive visualization of the hub-and-spoke network"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Hub-and-Spoke Network Design Analysis', fontsize=16, fontweight='bold')
    
    # 1. Network topology visualization
    ax1.set_title('Network Topology', fontweight='bold')
    
    # Plot all nodes
    for i in range(network.num_nodes):
        x, y = network.coordinates[i]
        
        if i in solution['hub_locations']:
            # Plot hub
            ax1.scatter(x, y, s=300, c='red', marker='s', 
                      edgecolors='black', linewidth=2, label='Hub' if i == solution['hub_locations'][0] else '')
            ax1.text(x+1, y+1, f'H{i+1}', fontweight='bold', fontsize=10)
        else:
            # Plot spoke
            ax1.scatter(x, y, s=100, c='lightblue', 
                      edgecolors='black', linewidth=1, label='Spoke' if i == 0 else '')
            ax1.text(x+1, y+1, f'{i+1}', fontsize=8)
    
    # Plot hub connections
    for i, hub1 in enumerate(solution['hub_locations']):
        for hub2 in solution['hub_locations'][i+1:]:
            x1, y1 = network.coordinates[hub1]
            x2, y2 = network.coordinates[hub2]
            ax1.plot([x1, x2], [y1, y2], '-', color='green', 
                    linewidth=2, alpha=0.6, label='Inter-hub' if i == 0 else '')
    
    # Plot spoke assignments
    for node, hub in solution['assignments'].items():
        if node != hub:
            node_x, node_y = network.coordinates[node]
            hub_x, hub_y = network.coordinates[hub]
            ax1.plot([node_x, hub_x], [node_y, hub_y], '--', 
                    color='gray', alpha=0.4, linewidth=1)
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # 2. Cost breakdown
    ax2.set_title('Cost Breakdown', fontweight='bold')
    
    costs = ['Fixed Cost', 'Variable Cost']
    values = [metrics['fixed_cost'], metrics['variable_cost']]
    colors = ['#FF6B6B', '#4ECDC4']
    
    bars = ax2.bar(costs, values, color=colors, alpha=0.8)
    ax2.set_ylabel('Cost ($)')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + max(values)*0.01,
                f'${value:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    # 3. Hub utilization
    ax3.set_title('Hub Utilization', fontweight='bold')
    
    hub_names = [f'Hub {h+1}' for h in solution['hub_locations']]
    utilizations = [metrics['hub_utilization'][h] for h in solution['hub_locations']]
    capacities = [network.hub_capacities[h] for h in solution['hub_locations']]
    
    x_pos = np.arange(len(hub_names))
    bars = ax3.bar(x_pos, utilizations, color='#FFD93D', alpha=0.8)
    
    ax3.set_xlabel('Hub')
    ax3.set_ylabel('Utilization (%)')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(hub_names)
    ax3.grid(True, alpha=0.3)
    
    # Add capacity labels
    for i, (bar, capacity) in enumerate(zip(bars, capacities)):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                f'Cap: {capacity}', ha='center', va='bottom', fontsize=8)
    
    # 4. Demand matrix heatmap
    ax4.set_title('Demand Matrix Heatmap', fontweight='bold')
    
    # Create a smaller demand matrix for visualization
    demand_viz = network.demand_matrix[:10, :10]  # Show first 10x10 for clarity
    
    im = ax4.imshow(demand_viz, cmap='YlOrRd', aspect='auto')
    ax4.set_xlabel('Destination Node')
    ax4.set_ylabel('Origin Node')
    ax4.set_xticks(range(10))
    ax4.set_yticks(range(10))
    ax4.set_xticklabels(range(1, 11))
    ax4.set_yticklabels(range(1, 11))
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax4)
    cbar.set_label('Demand Volume')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualization
fig = visualize_hub_spoke_network(network, solution, metrics)

In [ ]:
# Sensitivity analysis
def perform_sensitivity_analysis(network):
    """Perform sensitivity analysis on key parameters"""
    
    print("=== SENSITIVITY ANALYSIS ===")
    
    # Test different alpha values
    alpha_values = [0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1.0]
    alpha_results = []
    
    for alpha in alpha_values:
        network.alpha = alpha
        sol, _ = network.solve_mip_model(max_hubs=5)
        met = network.calculate_solution_metrics(sol)
        
        alpha_results.append({
            'alpha': alpha,
            'total_cost': met['total_cost'],
            'hub_count': met['hub_count'],
            'avg_path_length': met['avg_path_length']
        })
    
    # Test different max_hubs values
    hub_counts = [2, 3, 4, 5, 6, 7, 8]
    hub_results = []
    
    # Reset alpha to original
    network.alpha = 0.75
    
    for max_hubs in hub_counts:
        sol, _ = network.solve_mip_model(max_hubs=max_hubs)
        met = network.calculate_solution_metrics(sol)
        
        hub_results.append({
            'max_hubs': max_hubs,
            'actual_hubs': met['hub_count'],
            'total_cost': met['total_cost'],
            'fixed_cost': met['fixed_cost'],
            'variable_cost': met['variable_cost']
        })
    
    return alpha_results, hub_results

# Perform sensitivity analysis
alpha_results, hub_results = perform_sensitivity_analysis(network)

# Create sensitivity analysis visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Sensitivity Analysis', fontsize=16, fontweight='bold')

# 1. Alpha sensitivity - Total cost
ax1.set_title('Impact of Alpha on Total Cost', fontweight='bold')
ax1.plot([r['alpha'] for r in alpha_results], 
         [r['total_cost'] for r in alpha_results], 
         'o-', linewidth=2, markersize=8, color='#FF6B6B')
ax1.set_xlabel('Alpha (Inter-hub Discount Factor)')
ax1.set_ylabel('Total Cost ($)')
ax1.grid(True, alpha=0.3)

# 2. Alpha sensitivity - Hub count
ax2.set_title('Impact of Alpha on Hub Count', fontweight='bold')
ax2.plot([r['alpha'] for r in alpha_results], 
         [r['hub_count'] for r in alpha_results], 
         'o-', linewidth=2, markersize=8, color='#4ECDC4')
ax2.set_xlabel('Alpha (Inter-hub Discount Factor)')
ax2.set_ylabel('Number of Hubs')
ax2.grid(True, alpha=0.3)

# 3. Max hubs sensitivity - Total cost
ax3.set_title('Impact of Max Hubs on Total Cost', fontweight='bold')
ax3.plot([r['max_hubs'] for r in hub_results], 
         [r['total_cost'] for r in hub_results], 
         'o-', linewidth=2, markersize=8, color='#FFD93D')
ax3.set_xlabel('Maximum Number of Hubs')
ax3.set_ylabel('Total Cost ($)')
ax3.grid(True, alpha=0.3)

# 4. Cost breakdown by max hubs
ax4.set_title('Cost Components vs Max Hubs', fontweight='bold')
max_hubs_vals = [r['max_hubs'] for r in hub_results]
fixed_costs = [r['fixed_cost'] for r in hub_results]
variable_costs = [r['variable_cost'] for r in hub_results]

ax4.plot(max_hubs_vals, fixed_costs, 'o-', linewidth=2, markersize=6, 
         label='Fixed Cost', color='#FF6B6B')
ax4.plot(max_hubs_vals, variable_costs, 'o-', linewidth=2, markersize=6, 
         label='Variable Cost', color='#4ECDC4')
ax4.set_xlabel('Maximum Number of Hubs')
ax4.set_ylabel('Cost ($)')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key insights
print(f"\n=== KEY INSIGHTS ===")
print(f"1. Alpha Impact:")
print(f"   - Lower alpha (more discount) encourages more hubs")
print(f"   - Higher alpha (less discount) encourages fewer hubs")
print(f"   - Optimal alpha balances hub costs vs transportation costs")

print(f"\n2. Hub Count Impact:")
print(f"   - More hubs = higher fixed costs, lower variable costs")
print(f"   - Fewer hubs = lower fixed costs, higher variable costs")
print(f"   - Optimal hub count depends on demand distribution")

### Why this Tier exists vs earlier Tiers
Tier 1 provides the mathematical foundation for hub-and-spoke network design:
- **Optimal solutions**: Provides provably optimal solutions using mixed-integer programming
- **Mathematical rigor**: Establishes the formal problem formulation and constraints
- **Benchmark baseline**: Serves as the gold standard for comparing heuristic methods
- **Sensitivity analysis**: Enables systematic analysis of parameter impacts
- **Theoretical foundation**: Provides the mathematical basis for all subsequent tiers

### Pros / Cons vs earlier Tiers
**Pros vs Heuristic Methods (Tiers 2-3):**
- **Optimality guarantee**: Finds the mathematically optimal solution
- **Systematic analysis**: Enables sensitivity analysis and parameter tuning
- **Reproducible results**: Same input always produces same optimal output
- **Constraint handling**: Properly handles all constraints and requirements
- **Performance benchmark**: Provides baseline for evaluating other methods

**Cons:**
- **Computational complexity**: Can be slow for large problem instances
- **Scalability limits**: May not scale well to very large networks
- **Solver dependency**: Requires specialized optimization software
- **Linearization needs**: May require problem simplification for tractability
- **Memory intensive**: Can require significant memory for large formulations

### When to use this Tier
- **Small to medium networks**: Where optimality is important and computation time is acceptable
- **Benchmark studies**: When evaluating the performance of heuristic methods
- **Regulatory environments**: Where provable optimality may be required
- **Strategic planning**: For high-stakes network design decisions
- **Parameter analysis**: When studying the impact of different parameters on solutions
- **Academic research**: For theoretical analysis and algorithm development
- **Baseline comparison**: When developing and testing new optimization methods

### Key Insights from the Analysis

The mathematical formulation reveals several important insights:

1. **Trade-off balance**: The optimal solution balances fixed hub costs against variable transportation costs
2. **Alpha sensitivity**: The inter-hub discount factor significantly impacts hub selection and network structure
3. **Economies of scale**: Hub consolidation provides cost benefits but increases transportation distances
4. **Capacity constraints**: Hub capacity limits can significantly impact optimal network design
5. **Demand patterns**: The distribution of demand between node pairs influences optimal hub locations

This mathematical foundation provides the essential baseline for understanding hub-and-spoke network design and serves as the benchmark against which all other optimization methods are measured. The systematic approach ensures optimal solutions while providing deep insights into the fundamental trade-offs in network design.